In [7]:
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse import csr_matrix, hstack
from scipy.sparse.csgraph import laplacian
from scipy.sparse.linalg import eigsh
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor

In [12]:
DATA_PATH = "data/train_lang.csv"

TEXT_COL = "sentence"
RATING_COL = "label"
LANG_COL = "lang"

df = pd.read_csv(DATA_PATH)
df = df[[TEXT_COL, RATING_COL, LANG_COL]].dropna()
df[TEXT_COL] = df[TEXT_COL].astype(str)
df[RATING_COL] = df[RATING_COL].astype(int)

df["strat"] = df[LANG_COL].astype(str) + "_" + df[RATING_COL].astype(str)

train_df, val_df = train_test_split(
    df,
    test_size=0.10,
    random_state=42,
    stratify=df["strat"]
)

y_train = train_df[RATING_COL].values.astype(float)
y_val = val_df[RATING_COL].values.astype(float)

print(train_df.shape, val_df.shape)

(226800, 4) (25200, 4)


In [9]:
from sklearn.model_selection import train_test_split

df["strat"] = df[LANG_COL].astype(str) + "_" + df[RATING_COL].astype(str)

train_df, val_df = train_test_split(
    df,
    test_size=0.10,
    random_state=42,
    stratify=df["strat"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(train_df.shape, val_df.shape)
print("Train rating distribution:")
print(train_df[RATING_COL].value_counts(normalize=True).sort_index())
print("Val rating distribution:")
print(val_df[RATING_COL].value_counts(normalize=True).sort_index())
print("Train language distribution:")
print(train_df[LANG_COL].value_counts(normalize=True))
print("Val language distribution:")
print(val_df[LANG_COL].value_counts(normalize=True))

(226800, 4) (25200, 4)
Train rating distribution:
label
0    0.2
1    0.2
2    0.2
3    0.2
4    0.2
Name: proportion, dtype: float64
Val rating distribution:
label
0    0.2
1    0.2
2    0.2
3    0.2
4    0.2
Name: proportion, dtype: float64
Train language distribution:
lang
deu_Latn    0.501389
eng_Latn    0.498611
Name: proportion, dtype: float64
Val language distribution:
lang
deu_Latn    0.501389
eng_Latn    0.498611
Name: proportion, dtype: float64


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from scipy.sparse import hstack

y_train = train_df[RATING_COL].values.astype(float)
y_val = val_df[RATING_COL].values.astype(float)

word_tfidf = TfidfVectorizer(
    lowercase=True,
    analyzer="word",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    max_features=300_000,
    sublinear_tf=True
)

char_tfidf = TfidfVectorizer(
    lowercase=True,
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_df=0.95,
    max_features=300_000,
    sublinear_tf=True
)

Xw_train = word_tfidf.fit_transform(train_df[TEXT_COL])
Xw_val = word_tfidf.transform(val_df[TEXT_COL])

Xc_train = char_tfidf.fit_transform(train_df[TEXT_COL])
Xc_val = char_tfidf.transform(val_df[TEXT_COL])

X_train_base = hstack([Xw_train, Xc_train]).tocsr()
X_val_base = hstack([Xw_val, Xc_val]).tocsr()

ridge = Ridge(alpha=10.0, random_state=42)
ridge.fit(X_train_base, y_train)

pred_ridge = ridge.predict(X_val_base)
pred_ridge = np.clip(pred_ridge, 1, 5)

mae_ridge = mean_absolute_error(y_val, pred_ridge)
print("TF-IDF Ridge MAE:", mae_ridge)

TF-IDF Ridge MAE: 0.6726998603715488


In [ ]:
------------------------------

In [14]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

# Smaller vocab first
vec = CountVectorizer(
    lowercase=True,
    analyzer="word",
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.9,
    max_features=30_000,
    binary=True
)

Xtr_count = vec.fit_transform(train_df[TEXT_COL])
Xva_count = vec.transform(val_df[TEXT_COL])

vocab = np.array(vec.get_feature_names_out())
n_terms = len(vocab)

print("Vocab size:", n_terms)

Vocab size: 30000


In [15]:
# Term rating distributions
Y = np.zeros((len(train_df), 5), dtype=np.float32)
Y[np.arange(len(train_df)), y_train.astype(int) - 1] = 1.0

alpha = 1.0
term_counts = Xtr_count.T @ Y
P = np.asarray(term_counts, dtype=np.float32) + alpha
P /= P.sum(axis=1, keepdims=True)

CDF = np.cumsum(P, axis=1)
term_mean = P @ np.arange(1, 6)

In [16]:
# Build compact term descriptors
# These are tiny: one row per term, a few dimensions only.
term_freq = np.asarray(Xtr_count.sum(axis=0)).ravel().astype(np.float32)

Z = np.column_stack([
    P,
    CDF,
    term_mean,
    np.log1p(term_freq),
]).astype(np.float32)

# Standardize
Z = (Z - Z.mean(axis=0)) / (Z.std(axis=0) + 1e-6)

In [17]:
# KNN graph over terms
k = 30

nn = NearestNeighbors(
    n_neighbors=k + 1,
    metric="euclidean",
    algorithm="auto",
    n_jobs=-1
)

nn.fit(Z)
dist, ind = nn.kneighbors(Z)

rows = np.repeat(np.arange(n_terms), k)
cols = ind[:, 1:].reshape(-1)
dists = dist[:, 1:].reshape(-1)

sigma = np.median(dists)
weights = np.exp(-(dists ** 2) / (2 * sigma ** 2))

W = csr_matrix((weights, (rows, cols)), shape=(n_terms, n_terms))
W = 0.5 * (W + W.T)
W.eliminate_zeros()

print("Graph edges:", W.nnz)

Graph edges: 1130233


In [18]:
from scipy.sparse.csgraph import laplacian
from scipy.sparse.linalg import eigsh

dim = 64

L = laplacian(W, normed=True).astype(np.float32)

vals, vecs = eigsh(
    L,
    k=dim + 1,
    which="SM",
    tol=1e-3,
    maxiter=5000
)

order = np.argsort(vals)
E = vecs[:, order][:, 1:dim + 1].astype(np.float32)

print("Embedding matrix:", E.shape)
print("Small eigenvalues:", vals[order][:8])

Embedding matrix: (30000, 64)
Small eigenvalues: [-1.8245930e-07 -8.6610903e-08  2.3587004e-03  5.0276122e-03
  6.7621386e-03  1.1595809e-02  1.3079429e-02  1.6020382e-02]


In [19]:
tfidf_spec = TfidfVectorizer(
    vocabulary=vec.vocabulary_,
    lowercase=True,
    analyzer="word",
    ngram_range=(1, 2),
    sublinear_tf=True
)

Xtr_spec = tfidf_spec.fit_transform(train_df[TEXT_COL]) @ E
Xva_spec = tfidf_spec.transform(val_df[TEXT_COL]) @ E

In [20]:
# Add tiny extra features
def term_features(X):
    X = X.astype(np.float32)
    X.data[:] = 1.0
    cnt = np.asarray(X.sum(axis=1)).ravel()
    cnt_safe = np.maximum(cnt, 1)

    mean_score = np.asarray(X @ term_mean).ravel() / cnt_safe
    pos_frac = np.asarray(X @ (term_mean >= 4)).ravel() / cnt_safe
    neg_frac = np.asarray(X @ (term_mean <= 2)).ravel() / cnt_safe

    return np.vstack([cnt, mean_score, pos_frac, neg_frac]).T

Ftr = term_features(Xtr_count)
Fva = term_features(Xva_count)

Xtr = np.hstack([Xtr_spec, Ftr])
Xva = np.hstack([Xva_spec, Fva])

In [21]:
lgb = LGBMRegressor(
    objective="mae",
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    n_jobs=-1
)

lgb.fit(Xtr, y_train)

pred_spec = np.clip(lgb.predict(Xva), 1, 5)
mae_spec = mean_absolute_error(y_val, pred_spec)

print("OWSE Spectral MAE:", mae_spec)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.044586 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17276
[LightGBM] [Info] Number of data points in the train set: 226800, number of used features: 68
[LightGBM] [Info] Start training from score 2.000000


/home/micha/anaconda3/envs/cil_fresh/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


OWSE Spectral MAE: 0.6325777042816653
